In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        fname=os.path.join(dirname, filename)
        print(fname)
        if "sub" in filename:
            df_sub=pd.read_csv(fname)
        elif "train" in filename:
            df_train=pd.read_csv(fname)
        elif "test" in filename:
            df_test=pd.read_csv(fname)
        else:
            df_origin=pd.read_csv(fname)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/rainfall-prediction-using-machine-learning/Rainfall.csv
/kaggle/input/playground-series-s5e3/sample_submission.csv
/kaggle/input/playground-series-s5e3/train.csv
/kaggle/input/playground-series-s5e3/test.csv


In [2]:
# !pip install --upgrade scikit-learn==1.4 --no-cache-dir
# !pip install cmaes


In [3]:
import pandas as pd
import numpy as np
from functools import partial
from copy import deepcopy
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
# Import libraries for gradient boosting
import optuna
from optuna.samplers import CmaEsSampler
from sklearn.base import BaseEstimator, TransformerMixin
import xgboost as xgb
import lightgbm as lgb
# from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
# from sklearn.svm import NuSVC, SVC
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.linear_model import LogisticRegression
# from sklearn.neural_network import MLPRegressor
# from sklearn.gaussian_process import GaussianProcessRegressor
# from sklearn.gaussian_process.kernels import RBF
from catboost import CatBoost, CatBoostRegressor, CatBoostRegressor
from catboost import Pool
from category_encoders import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import log_loss, auc,roc_auc_score #root_mean_squared_error,
from sklearn.preprocessing import StandardScaler#,TargetEncoder
from sklearn.model_selection import StratifiedKFold, KFold


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from transformers import AdamW
from sklearn.model_selection import KFold, GroupKFold
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
device

device(type='cuda')

In [5]:
df_origin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   day                     366 non-null    int64  
 1   pressure                366 non-null    float64
 2   maxtemp                 366 non-null    float64
 3   temparature             366 non-null    float64
 4   mintemp                 366 non-null    float64
 5   dewpoint                366 non-null    float64
 6   humidity                366 non-null    int64  
 7   cloud                   366 non-null    int64  
 8   rainfall                366 non-null    object 
 9   sunshine                366 non-null    float64
 10           winddirection  365 non-null    float64
 11  windspeed               365 non-null    float64
dtypes: float64(8), int64(3), object(1)
memory usage: 34.4+ KB


In [6]:
df_origin.describe()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
count,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,365.000000,365.000000
mean,15.756831,1013.742623,26.191257,23.747268,21.894536,19.989071,80.177596,71.128415,4.419399,101.506849,21.536986
std,8.823592,6.414776,5.978343,5.632813,5.594153,5.997021,10.062470,21.798012,3.934398,81.723724,10.069712
min,1.000000,998.500000,7.100000,4.900000,3.100000,-0.400000,36.000000,0.000000,0.000000,10.000000,4.400000
25%,8.000000,1008.500000,21.200000,18.825000,17.125000,16.125000,75.000000,58.000000,0.500000,40.000000,13.700000
50%,16.000000,1013.000000,27.750000,25.450000,23.700000,21.950000,80.500000,80.000000,3.500000,70.000000,20.500000
75%,23.000000,1018.100000,31.200000,28.600000,26.575000,25.000000,87.000000,88.000000,8.200000,190.000000,27.900000
max,31.000000,1034.600000,36.300000,32.400000,30.000000,26.700000,98.000000,100.000000,12.100000,350.000000,59.500000


In [7]:
origin_std=df_origin.describe().iloc[2,]
origin_std

day                        8.823592
pressure                   6.414776
maxtemp                    5.978343
temparature                5.632813
mintemp                    5.594153
dewpoint                   5.997021
humidity                  10.062470
cloud                     21.798012
sunshine                   3.934398
         winddirection    81.723724
windspeed                 10.069712
Name: std, dtype: float64

In [8]:
index=origin_std.keys()
index = [ a.strip() for a in index ]
origin_std.index=index

In [9]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2190 entries, 0 to 2189
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             2190 non-null   int64  
 1   day            2190 non-null   int64  
 2   pressure       2190 non-null   float64
 3   maxtemp        2190 non-null   float64
 4   temparature    2190 non-null   float64
 5   mintemp        2190 non-null   float64
 6   dewpoint       2190 non-null   float64
 7   humidity       2190 non-null   float64
 8   cloud          2190 non-null   float64
 9   sunshine       2190 non-null   float64
 10  winddirection  2190 non-null   float64
 11  windspeed      2190 non-null   float64
 12  rainfall       2190 non-null   int64  
dtypes: float64(10), int64(3)
memory usage: 222.5 KB


In [10]:
train_std=df_train.drop(['id','rainfall'],axis=1).describe().iloc[2,]
train_std

day              105.203592
pressure           5.655366
maxtemp            5.654330
temparature        5.222410
mintemp            5.059120
dewpoint           5.288406
humidity           7.800654
cloud             18.026498
sunshine           3.626327
winddirection     80.002416
windspeed          9.898659
Name: std, dtype: float64

In [11]:
origin_train=pd.concat([origin_std,train_std],axis=1)
origin_train.columns=['origin_std','train_std']

In [12]:
origin_train['diff']=origin_train['origin_std']-origin_train['train_std']
origin_train

,origin_std,train_std,diff
day,8.823592,105.203592,-96.380001
pressure,6.414776,5.655366,0.759411
maxtemp,5.978343,5.654330,0.324012
temparature,5.632813,5.222410,0.410403
mintemp,5.594153,5.059120,0.535033
dewpoint,5.997021,5.288406,0.708615
humidity,10.062470,7.800654,2.261817
cloud,21.798012,18.026498,3.771514
sunshine,3.934398,3.626327,0.308071
winddirection,81.723724,80.002416,1.721309


In [13]:
df_train.describe()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
count,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000,2190.000000
mean,1094.500000,179.948402,1013.602146,26.365799,23.953059,22.170091,20.454566,82.036530,75.721918,3.744429,104.863151,21.804703,0.753425
std,632.342866,105.203592,5.655366,5.654330,5.222410,5.059120,5.288406,7.800654,18.026498,3.626327,80.002416,9.898659,0.431116
min,0.000000,1.000000,999.000000,10.400000,7.400000,4.000000,-0.300000,39.000000,2.000000,0.000000,10.000000,4.400000,0.000000
25%,547.250000,89.000000,1008.600000,21.300000,19.300000,17.700000,16.800000,77.000000,69.000000,0.400000,40.000000,14.125000,1.000000
50%,1094.500000,178.500000,1013.000000,27.800000,25.500000,23.850000,22.150000,82.000000,83.000000,2.400000,70.000000,20.500000,1.000000
75%,1641.750000,270.000000,1017.775000,31.200000,28.400000,26.400000,25.000000,88.000000,88.000000,6.800000,200.000000,27.900000,1.000000
max,2189.000000,365.000000,1034.600000,36.000000,31.500000,29.800000,26.700000,98.000000,100.000000,12.100000,300.000000,59.500000,1.000000


# Feature Engineering

**Statistics**

In [14]:
#Add moving average 4
df_train['temp_mean_4'] = df_train['temparature'].rolling(window=4).mean().bfill().ffill()
df_train['windirection_mean_4'] = df_train['winddirection'].rolling(window=4).mean().bfill().ffill()
df_train['humidity_mean_4'] = df_train['humidity'].rolling(window=4).mean().bfill().ffill()
df_train['cloud_mean_4']=df_train['cloud'].rolling(window=4).mean().bfill().ffill()
df_train['windspeed_mean_4']=df_train['windspeed'].rolling(window=4).mean().bfill().ffill()


In [15]:
df_train['range_temp']=df_train['maxtemp']-df_train['mintemp']
df_train['range_temp_mean']=df_train['range_temp'].rolling(window=4).mean().bfill().ffill()

#do timeseries shift 
df_train['pressure_shift']=df_train['pressure'].shift(1).bfill()
df_train['humidity_shift']=df_train['humidity'].shift(1).bfill()
df_train['cloud_shift']=df_train['cloud'].shift(1).bfill()


In [16]:
df_test['temp_mean_4'] = df_test['temparature'].rolling(window=4).mean().bfill().ffill()
df_test['windirection_mean_4'] = df_test['winddirection'].rolling(window=4).mean().bfill().ffill()
df_test['humidity_mean_4'] = df_test['humidity'].rolling(window=4).mean().bfill().ffill()
df_test['cloud_mean_4']=df_test['cloud'].rolling(window=4).mean().bfill().ffill()
df_test['windspeed_mean_4']=df_test['windspeed'].rolling(window=4).mean().bfill().ffill()


In [17]:
df_test['range_temp']=df_test['maxtemp']-df_test['mintemp']
df_test['range_temp_mean']=df_test['range_temp'].rolling(window=4).mean().bfill().ffill()

#do timeseries shift 
df_test['pressure_shift']=df_test['pressure'].shift(1).bfill()
df_test['humidity_shift']=df_test['humidity'].shift(1).bfill()
df_test['cloud_shift']=df_test['cloud'].shift(1).bfill()


**Sinc and Cosine transformation for cyclic**

In [18]:
df_train['monthly_sin'] = np.sin(2 * np.pi * df_train['day'] / 30.44)
df_train['monthly_cos'] = np.cos(2 * np.pi * df_train['day'] / 30.44)
df_train['quarterly_sin'] = np.sin(2 * np.pi * df_train['day'] / 91.31)
df_train['quarterly_cos'] = np.cos(2 * np.pi * df_train['day'] / 91.31)
df_train['yearly_sin'] = np.sin(2 * np.pi * df_train['day'] / 365.25)
df_train['yearly_cos'] = np.cos(2 * np.pi * df_train['day'] / 365.25)



In [19]:
df_test['monthly_sin'] = np.sin(2 * np.pi * df_test['day'] / 30.44)
df_test['monthly_cos'] = np.cos(2 * np.pi * df_test['day'] / 30.44)
df_test['quarterly_sin'] = np.sin(2 * np.pi * df_test['day'] / 91.31)
df_test['quarterly_cos'] = np.cos(2 * np.pi * df_test['day'] / 91.31)
df_test['yearly_sin'] = np.sin(2 * np.pi * df_test['day'] / 365.25)
df_test['yearly_cos'] = np.cos(2 * np.pi * df_test['day'] / 365.25)


In [20]:
#Select columns to be dropped
drop_cols=['id','temparature','winddirection','windspeed','range_temp']
df_train=df_train.drop(drop_cols,axis=1)
df_test=df_test.drop(drop_cols,axis=1)

In [21]:
sc = StandardScaler()
Y_train = df_train['rainfall']
X_train=sc.fit_transform(df_train.drop('rainfall',axis=1))
X_test=sc.fit_transform(df_test)
X_val = X_train[-365:,]
Y_val = Y_train[-365:,]
X_train=X_train[:-365,]
Y_train=Y_train[:-365,]



In [22]:
from sklearn.model_selection import TimeSeriesSplit

class Rainfall_TS_Dataset(Dataset):
    def __init__(self, X, y, sequence_length, n_splits=5,device='cpu'):
        """
        Args:
            X (np.array): Feature data of shape (num_samples, num_features).
            y (np.array): Label data of shape (num_samples,).
            sequence_length (int): Length of each sequence.
            n_splits (int): Number of splits for TimeSeriesSplit.
        """
        self.X = X
        self.y = y
        self.sequence_length = sequence_length
        self.n_splits = n_splits
        
        # Create sequences
        self.X_sequences = np.array([X[i:i+sequence_length] for i in range(len(X) - sequence_length)])
        self.y_sequences = np.array([y[i+sequence_length-1] for i in range(len(y) - sequence_length)])
        
        # Initialize TimeSeriesSplit
        self.tscv = TimeSeriesSplit(n_splits=n_splits)
        self.splits = list(self.tscv.split(self.X_sequences))
        self.current_fold = 0
        self.device=device
    
    def set_fold(self, fold):
        """Set the current fold for training/validation."""
        if fold >= self.n_splits:
            raise ValueError(f"Fold {fold} is out of range. Maximum folds: {self.n_splits}")
        self.current_fold = fold
    
    def __len__(self):
        """Return the number of samples in the current fold."""
        train_index, val_index = self.splits[self.current_fold]
        return len(train_index)  # Return size of training set
    
    def __getitem__(self, idx):
        """Return a batch of sequences and labels for the current fold."""
        train_index, val_index = self.splits[self.current_fold]
        X_train = self.X_sequences[train_index]
        y_train = self.y_sequences[train_index]
        
        # Convert to PyTorch tensors
        X_train_tensor = torch.tensor(X_train[idx], dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train[idx], dtype=torch.float32)
        # Add extra dimension for label if necessary
        if y_train_tensor.dim() == 0:  # If scalar
            y_train_tensor = y_train_tensor.unsqueeze(0)  # Add dimension at index 0
        else:
            y_train_tensor = y_train_tensor.unsqueeze(1)  # Add dimension at index 1
        
        
        return X_train_tensor.to(self.device), y_train_tensor.to(self.device)

    def get_validation_data(self):
        """Return the validation set for the current fold."""
        _, val_index = self.splits[self.current_fold]
        X_val = self.X_sequences[val_index]
        y_val = self.y_sequences[val_index]
        
        # Convert to PyTorch tensors
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
        # Add extra dimension for label if necessary
        if y_val_tensor.dim() == 0:  # If scalar
            y_val_tensor = y_val_tensor.unsqueeze(0)  # Add dimension at index 0
        else:
            y_val_tensor = y_val_tensor.unsqueeze(1)  # Add dimension at index 1
        
        
        return X_val_tensor.to(self.device), y_val_tensor.to(self.device)

In [23]:
class RainfallTest(Dataset):

    def __init__(self,X_test,device='cpu'):
        # Initialize data, download, etc.
        # read with numpy or pandas
        #xy = np.loadtxt('./data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = X_test.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(X_test).float() # size [n_samples, n_features]
        self.labels = df_test.columns
        self.device = device
        
    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index].to(self.device)

    def getlabels(self, index):
        return self.labels[index]
    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [24]:
class RainfallValidate(Dataset):

    def __init__(self):
        # Initialize data, download, etc.
        # read with numpy or pandas
        #xy = np.loadtxt('./data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = X_val.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(X_val).float() # size [n_samples, n_features]
        self.labels = df_train.drop('rainfall',axis=1).columns
        self.y_data = torch.from_numpy(Y_val.values).float() # size [n_samples, 1]
        self.y_label = 'rainfall'

    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def getlabels(self, index):
        return self.labels[index]
    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [25]:
class RainfallDataset(Dataset):

    def __init__(self):
        # Initialize data, download, etc.
        # read with numpy or pandas
        #xy = np.loadtxt('./data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = X_train.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(X_train).float() # size [n_samples, n_features]
        self.labels = df_train.drop('rainfall',axis=1).columns
        self.y_data = torch.from_numpy(Y_train.values).float() # size [n_samples, 1]
        self.y_label = 'rainfall'

    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def getlabels(self, index):
        return self.labels[index]
    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [26]:
#Use dataloader
batch_size=12
n_splits=5
train_dataset = Rainfall_TS_Dataset(X_train,Y_train,7,n_splits)
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=False,
                          num_workers=1)

# convert to an iterator and look at one random sample


In [27]:
import torch.optim as optim

class Rainfall_LSTM(nn.Module):
    def __init__(self, input_dim=22, hidden_dim=128, output_dim=1, num_layers=2, dropout=0.3):
        super(Rainfall_LSTM, self).__init__()
        
        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_dim,  # 22 input features
            hidden_size=hidden_dim,  # Hidden state size
            num_layers=num_layers,   # Number of LSTM layers
            batch_first=True,        # Input shape: (batch, seq_len, input_dim)
            dropout=dropout,         # Dropout for regularization
            bidirectional=False      # Single-direction LSTM
        )
        
        # Batch normalization after LSTM
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        
        # Fully connected layer to map LSTM output to 1 output
        self.fc = nn.Linear(hidden_dim, output_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Input x has shape (batch_size, 22)
        # Add a sequence dimension to make it (batch_size, 1, 22)
        #x = x.unsqueeze(1)  # Shape: (batch_size, 1, 22)
        
        # LSTM layer
        lstm_out, _ = self.lstm(x)  # lstm_out shape: (batch_size, 1, hidden_dim)
        
        # Take the output of the last time step
        lstm_out = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_dim)
        
        # Batch normalization (skip if batch size is 1 during training)
        if lstm_out.size(0) > 1 or not self.training:
            lstm_out = self.bn1(lstm_out)
        
        # Dropout
        lstm_out = self.dropout(lstm_out)
        
        # Fully connected layer
        output = self.fc(lstm_out)  # Shape: (batch_size, 1)
        
        return output  # Return raw output 

In [28]:
test_dataset=RainfallTest(X_test,device)
test_dataloader=DataLoader(test_dataset,batch_size=365,shuffle=False)
features = next(iter(test_dataloader))
#model.predict(features)

In [29]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import CosineAnnealingLR
input_dim = X_train.shape[1]
hidden_dim = 180
output_dim = 1
num_layers = 2
dropout = 0.29
#batch_size = 365
learning_rate = 0.006
num_epochs = 400
patience=5
weight_decay = 1e-3 # L2 regularization strength

# Initialize model, loss, and optimizer
model = Rainfall_LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers, dropout=dropout)
model=model.to(device)
criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Initialize learning rate scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=patience)

# Initialize CosineAnnealingLR scheduler
#scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)
# Create DataLoader
dataloader = train_loader


# Initialize variables to track the best model
best_loss = float('inf')  # For regression (lower is better)
# best_accuracy = 0.0  # For classification (higher is better)
best_model_state = None

# Training loop
for epoch in range(num_epochs):
   # model.train()  # Set model to training mode
    
    total_val_loss = 0.0
    for fold in range(n_splits):
        #print(f"Training on Fold {fold + 1}")
        train_dataset.set_fold(fold)  # Set the current fold
        model.train()
        for batch_idx, batch in enumerate(dataloader):
            inputs, targets = batch  # features is a list of 23 tensors, targets is a list of 1 tensors
            
            # Stack features and targets into batches
            #features = torch.stack(features)  # Shape: (23, 22)
            #targets = torch.stack(targets)    # Shape: (23, 1)
            inputs = inputs.to(device)
            targets = targets.to(device)
            #inputs = inputs.float()  # Convert inputs to float32
            #targets = targets.float()  # Convert targets to float32
    
            outputs = model(inputs)  # Shape: (batch_size, output_dim)
            loss = criterion(outputs, targets.reshape(-1,1))  # Compute loss
            
            # Backward pass and optimization
            optimizer.zero_grad()  # Clear gradients
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights
    
        # Validation phase
        model.eval()  # Set model to evaluation mode
        val_loss = 0.0
        with torch.no_grad():  # Disable gradient computation
            X_val_tensor, y_val_tensor = train_dataset.get_validation_data()  # Get validation set
            X_val_tensor=X_val_tensor.to(device)
            y_val_tensor=y_val_tensor.to(device)
            
            outputs = model(X_val_tensor)  # Shape: (batch_size, 1)
                
            # Compute loss
            loss = criterion(outputs, y_val_tensor)  # Regression or binary classification loss
            total_val_loss += loss.item()
            #print(f"Epoch [{epoch+1}], Fold [{fold+1}/{n_splits}], Validation Loss: {val_loss:.4f}")
        
    # Compute average validation loss
    avg_val_loss = total_val_loss / n_splits
    if (epoch+1)%20==0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss:.4f}")
    
    # Update learning rate scheduler
    scheduler.step(avg_val_loss)  # Pass validation loss to scheduler
    #scheduler.step()
    
    # Save the best model
    if avg_val_loss < best_loss:  # For regression (lower is better)
        print(f"Validation loss improved from {best_loss:.4f} to {avg_val_loss:.4f}. Saving model...")
        best_loss = avg_val_loss
        best_model_state = model.state_dict()  # Save the model's state dictionary
        torch.save(best_model_state, "best_model.pth")  # Save to file

# Load the best model
model.load_state_dict(torch.load("best_model.pth"))
print(f"Best model loaded. with best loss {best_loss}")
print("Training complete!")

Validation loss improved from inf to 0.5096. Saving model...
Validation loss improved from 0.5096 to 0.4394. Saving model...
Validation loss improved from 0.4394 to 0.4084. Saving model...
Validation loss improved from 0.4084 to 0.3922. Saving model...
Validation loss improved from 0.3922 to 0.3907. Saving model...
Validation loss improved from 0.3907 to 0.3896. Saving model...
Epoch [20/400], Validation Loss: 0.4273
Validation loss improved from 0.3896 to 0.3697. Saving model...
Validation loss improved from 0.3697 to 0.3478. Saving model...
Validation loss improved from 0.3478 to 0.3336. Saving model...
Validation loss improved from 0.3336 to 0.3284. Saving model...
Epoch [40/400], Validation Loss: 0.3415
Epoch [60/400], Validation Loss: 0.3408
Epoch [80/400], Validation Loss: 0.3408
Epoch [100/400], Validation Loss: 0.3434
Epoch [120/400], Validation Loss: 0.3466
Epoch [140/400], Validation Loss: 0.3409
Epoch [160/400], Validation Loss: 0.3429
Epoch [180/400], Validation Loss: 0.344

<ipython-input-29-bf10a28dd7d5>:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth"))


In [30]:
model.eval()

features=features.unsqueeze(1).to(device)
with torch.no_grad():
    logits = model(features)  # Shape: (1, 1)

# Convert logits to probabilities using sigmoid
probabilities = torch.sigmoid(logits)  # Shape: (1, 1)
threshold = 0.5
binary_output1 = (probabilities>threshold).float()

#print("Logits:", logits)
#print("Probabilities:", probabilities)

In [31]:
features = next(iter(test_dataloader))
with torch.no_grad():
    logits = model(features.unsqueeze(1).to(device))  # Shape: (1, 1)

# Convert logits to probabilities using sigmoid
probabilities = torch.sigmoid(logits)  # Shape: (1, 1)
threshold = 0.5
binary_output2 = (probabilities>threshold).float()


In [32]:
if binary_output1.device.type=='cpu':
    temp=np.concatenate((binary_output1,binary_output2))
else:
    temp=np.concatenate((binary_output1.cpu(),binary_output2.cpu()))
temp.shape

(730, 1)

In [33]:
temp2 = pd.concat([df_sub.drop('rainfall',axis=1),pd.DataFrame(temp,columns=['rainfall'])],axis=1)

In [34]:
temp2.to_csv("submission_ts_x_gpu.csv",index=False)